In [2]:
import torch
import subprocess

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("CUDA version:", torch.version.cuda)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("VRAM:",
          round(torch.cuda.get_device_properties(0).total_memory / 1e9, 2),
          "GB")


PyTorch: 2.14.1+cu130
CUDA available: True
CUDA version: 13.0
GPU: Tesla T4
VRAM: 15.64 GB


In [4]:
!pip install -U uv

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.4/17.4 MB 39.8 MB/s  0:00:00


In [7]:
!uv pip install --system vllm --torch-backend=cu130
!nvidia-smi


Using Python 3.13.16 environment at: /usr
Checked 1 package in 104ms
Mon Nov 16 11:26:36 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   34C    P8              9W /   70W |       3MiB /  15360MiB |      0%      Default |
|                                         |                        |   

In [9]:
import torch
import vllm
print("vllm version:",vllm.__version__)

vllm version: 0.33.0


In [11]:
!pip install -q huggingface_hub

In [17]:
from huggingface_hub import notebook_login
notebook_login()


In [19]:

from vllm import LLM

model_id = "Qwen/Qwen2.5-1.5B-Instruct"

llm = LLM(
    model=model_id,
    dtype="half",
    max_model_len=2048,
    gpu_memory_utilization=0.85,
    max_num_seqs=4
)

print("Qwen baseline model loaded successfully!")


(MainProcess pid=516) INFO 11-16 11:58:51 [startup.py:96] non-default args: {'dtype': 'half', 'max_model_len': 2048, 'gpu_memory_utilization': 0.85, 'max_num_seqs': 4, 'disable_log_stats': True, 'model': 'Qwen/Qwen2.5-1.5B-Instruct'}


config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

(MainProcess pid=516) INFO 11-16 11:59:16 [model.py:744] Resolved architecture: Qwen2ForCausalLM
(MainProcess pid=516) WARNING 11-16 11:59:16 [model.py:2516] Casting torch.bfloat16 to torch.float16.
(MainProcess pid=516) INFO 11-16 11:59:16 [model.py:2170] Using max model len 2048
(MainProcess pid=516) INFO 11-16 11:59:16 [scheduler.py:310] Chunked prefill is enabled with max_num_batched_tokens=8192.
(MainProcess pid=516) INFO 11-16 11:59:16 [kernel.py:480] Final IR op priority after setting platform defaults: IrOpPriorityConfig(rms_norm=['native'], fused_add_rms_norm=['native'], gelu_and_mul_sparse=['triton', 'native'])


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 9 files:   0%|          | 0/9 [00:00<?, ?it/s]

(MainProcess pid=516) WARNING 11-16 11:59:20 [system_utils.py:154] We must use the `spawn` multiprocessing start method. Overriding VLLM_WORKER_MULTIPROC_METHOD to 'spawn'. See https://docs.vllm.ai/en/latest/usage/troubleshooting.html#python-multiprocessing for more information. Reasons: CUDA is initialized
(MainProcess pid=516) INFO 11-16 12:01:33 [hf.py:736] Detected the chat template content format to be 'string'. You can set `--chat-template-content-format` to override this.
Qwen baseline model loaded successfully!


In [20]:

import time
import pandas as pd
import torch
import subprocess

# Benchmark prompts
prompts = [
    "Explain artificial intelligence in simple words.",
    "What is model quantization and why is it useful?",
    "Explain how GPUs accelerate machine learning.",
    "What is the difference between FP16 and INT4?"
]

# Generation settings
sampling_params = SamplingParams(
    temperature=0,
    max_tokens=128,
    ignore_eos=True
)

# Warm-up (not included in measurements)
print("Warming up model...")
llm.generate([prompts[0]], SamplingParams(max_tokens=32))

# Batch throughput benchmark
print("Running baseline benchmark...")

start = time.perf_counter()
outputs = llm.generate(prompts, sampling_params)
batch_time = time.perf_counter() - start

total_tokens = sum(
    len(output.outputs[0].token_ids)
    for output in outputs
)

throughput = total_tokens / batch_time

# Individual request latency benchmark
latencies = []

for prompt in prompts:
    start = time.perf_counter()
    result = llm.generate([prompt], sampling_params)
    latencies.append(time.perf_counter() - start)

avg_latency = sum(latencies) / len(latencies)

# GPU memory usage
gpu_memory = subprocess.check_output(
    ["nvidia-smi", "--query-gpu=memory.used",
     "--format=csv,noheader,nounits"]
).decode().strip()

# Display results
print("\n========== FP16 BASELINE RESULTS ==========")
print("Model: Qwen2.5-1.5B-Instruct")
print("Precision: FP16")
print("Total generated tokens:", total_tokens)
print("Batch generation time:", round(batch_time, 3), "seconds")
print("Throughput:", round(throughput, 2), "tokens/sec")
print("Average single-request latency:",
      round(avg_latency, 3), "seconds")
print("GPU memory used:", gpu_memory, "MiB")

# Save results
baseline = pd.DataFrame([{
    "model": "Qwen2.5-1.5B-Instruct",
    "precision": "FP16",
    "total_tokens": total_tokens,
    "batch_time_sec": batch_time,
    "throughput_tokens_sec": throughput,
    "avg_request_latency_sec": avg_latency,
    "gpu_memory_mib": int(gpu_memory)
}])

baseline.to_csv("baseline_results.csv", index=False)

print("\nSaved: baseline_results.csv")


Warming up model...


Rendering prompts:   0%|          | 0/1 [00:00<?, ?it/s]

Processed prompts: 100%|██████████| 1/1 [00:01<00:00,  1.79s/it, est. speed input: 4.47 toks/s, output: 17.88 toks/s]

Running baseline benchmark...


Rendering prompts:   0%|          | 0/4 [00:00<?, ?it/s]

Processed prompts: 100%|██████████| 4/4 [00:01<00:00,  2.03it/s, est. speed input: 19.77 toks/s, output: 259.58 toks/s]


... (5 outputs left out)

Processed prompts: 100%|██████████| 1/1 [00:01<00:00,  1.88s/it, est. speed input: 4.25 toks/s, output: 68.01 toks/s]


Rendering prompts:   0%|          | 0/1 [00:00<?, ?it/s]

Processed prompts: 100%|██████████| 1/1 [00:01<00:00,  1.90s/it, est. speed input: 6.33 toks/s, output: 67.51 toks/s]


========== FP16 BASELINE RESULTS ==========
Model: Qwen2.5-1.5B-Instruct
Precision: FP16
Total generated tokens: 512
Batch generation time: 2.017 seconds
Throughput: 253.9 tokens/sec
Average single-request latency: 1.91 seconds
GPU memory used: 12337 MiB

Saved: baseline_results.csv


In [2]:
import gc
import torch

if "llm" in globals():
    del llm
    print("FP16 model variable removed.")
else:
    print("FP16 model variable not found. Already cleared.")

gc.collect()
torch.cuda.empty_cache()

print("GPU cache cleanup completed.")


FP16 model variable not found. Already cleared.
GPU cache cleanup completed.


In [3]:
from vllm import LLM

awq_model_id = "Qwen/Qwen2.5-1.5B-Instruct-AWQ"

awq_llm = LLM(
    model=awq_model_id,
    max_model_len=2048,
    gpu_memory_utilization=0.80,
    max_num_seqs=4
)

print("AWQ INT4 model loaded successfully!")

ModuleNotFoundError: No module named 'vllm'

In [4]:
import sys
import torch
import importlib.util

print("Python:", sys.version)
print("PyTorch:", torch.__version__)
print("CUDA:", torch.cuda.is_available())
print("vLLM installed:", importlib.util.find_spec("vllm") is not None)

Python: 3.13.18 (main, Oct 26 2026, 15:28:15) [GCC 13.3.0]
PyTorch: 2.14.1+cpu
CUDA: False
vLLM installed: False


In [5]:
!uv pip install --system vllm --torch-backend=cu130

Using Python 3.13.18 environment at: /usr
Resolved 198 packages in 2.05s
Prepared 103 packages in 1m 50s
Uninstalled 10 packages in 1.87s
Installed 103 packages in 1.33s
 + agent-detector==2.0.0
 + anthropic==2.8.0
 + apache-tvm-ffi==0.1.11
 + astor==0.8.1
 + blake3==1.0.11
 + cbor2==6.3.0
 + compressed-tensors==0.17.0
 + cuda-bindings==13.5.0
 + cuda-core==1.2.2
 + cuda-pathfinder==1.9.0
 + cuda-python==13.5.0
 + cuda-tile==1.8.1
 + cuda-toolkit==13.0.3.0
 + depyf==0.20.0
 + detect-installer==0.2.1
 + dnspython==2.9.2
 + email-validator==2.4.0
 - fastapi==0.141.1
 + fastapi==0.136.3
 + fastapi-cli==0.0.34
 + fastapi-cloud-cli==0.28.0
 + fastar==0.12.0
 + fastsafetensors==0.5.0
 + flashinfer-python==0.7.2.post1
 + humming-kernels==0.1.19
 + ijson==3.6.0
 + instanttensor==0.2.3
 + interegular==0.3.3
 + jmespath==1.1.0
 - lark==1.3.1
 + lark==1.2.2
 + llguidance==1.7.6
 - llvmlite==0.46.0
 + llvmlite==0.47.0
 + lm-format-enforcer==0.11.3
 + loguru==0.8.0
 + mcp==2.3.0
 + mcp-types==2.3.0

In [6]:
from vllm import LLM

awq_model_id = "Qwen/Qwen2.5-1.5B-Instruct-AWQ"

awq_llm = LLM(
    model=awq_model_id,
    max_model_len=2048,
    gpu_memory_utilization=0.80,
    max_num_seqs=4
)

print("AWQ INT4 model loaded successfully!")

W1116 12:16:43.648000 1919 torch/_opaque_base.py:6] torch._opaque_base is deprecated, use torch._custom_class_base instead
W1116 12:16:43.649000 1919 torch/_library/opaque_object.py:288] register_opaque_type is deprecated, use register_custom_class instead
W1116 12:16:43.652000 1919 torch/_library/opaque_object.py:211] typ='value' is deprecated, use typ='constant' instead


RuntimeError: operator torchvision::nms does not exist

In [7]:
from google.colab import files
files.download("baseline_results.csv")

FileNotFoundError: Cannot find file: baseline_results.csv

In [8]:
import pandas as pd

baseline = pd.DataFrame([{
    "model": "Qwen2.5-1.5B-Instruct",
    "precision": "FP16",
    "total_tokens": 512,
    "batch_time_sec": 2.017,
    "throughput_tokens_sec": 253.9,
    "avg_request_latency_sec": 1.91,
    "gpu_memory_mib": 12337
}])

baseline.to_csv("baseline_results.csv", index=False)

print(baseline)
print("\nBaseline CSV recreated!")

                   model precision  total_tokens  batch_time_sec  \
0  Qwen2.5-1.5B-Instruct      FP16           512           2.017   

   throughput_tokens_sec  avg_request_latency_sec  gpu_memory_mib  
0                  253.9                     1.91           12337  

Baseline CSV recreated!


In [3]:

!nvidia-smi

import torch

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("GPU memory:",
          round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2),
          "GB")


Mon Nov 16 17:16:13 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   32C    P8              9W /   70W |       3MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [4]:

from vllm import LLM, SamplingParams
import torch
import time
import pandas as pd

model = "Qwen/Qwen2.5-1.5B-Instruct-AWQ"

llm = LLM(
    model=model,
    quantization="awq",
    dtype="half",
    max_model_len=1024,
    max_num_seqs=2,
    gpu_memory_utilization=0.80
)

print("AWQ loaded on Tesla T4!")


ModuleNotFoundError: No module named 'vllm'

In [5]:

import sys

print("Python executable:", sys.executable)
print("Python version:", sys.version)

!{sys.executable} -m pip install -U vllm


Python executable: /usr/bin/python3
Python version: 3.13.16 (main, Oct  1 2026, 15:40:24) [GCC 13.3.0]
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 336.4/336.4 MB 19.2 MB/s  0:00:14
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.7/2.7 MB 83.6 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.1/62.1 MB 51.7 MB/s  0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 556.0/556.0 kB 29.0 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.8/3.8 MB 97.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 105.2 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 76.8 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.7/91.7 MB 58.2 MB/s  0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 92.0/92.0 MB 59.8 MB/s  0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 782.6/782.6 kB 43.1 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 78.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.2/43.2 MB 52.1 MB

In [6]:

import torch
import vllm
import sys

print("vLLM:", vllm.__version__)
print("PyTorch:", torch.__version__)
print("CUDA:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0))
print("Compute capability:", torch.cuda.get_device_capability(0))

!nvidia-smi


vLLM: 0.33.0
PyTorch: 2.14.1+cu130
CUDA: True
GPU: Tesla T4
Compute capability: (7, 5)
Mon Nov 16 17:26:04 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   32C    P8              9W /   70W |       3MiB /  15360MiB |      0%      Default |
|                                         |          

In [7]:

import torch
import time
from vllm import LLM, SamplingParams

model_name = "Qwen/Qwen2.5-1.5B-Instruct-AWQ"

print("Loading AWQ model on Tesla T4...")

llm = LLM(
    model=model_name,
    quantization="awq",
    dtype="half",
    max_model_len=512,
    max_num_seqs=2,
    gpu_memory_utilization=0.75,
    enforce_eager=True
)

print("SUCCESS: AWQ model loaded!")

# Quick inference test
sampling_params = SamplingParams(
    temperature=0,
    max_tokens=64
)

outputs = llm.generate(
    ["Explain large language model quantization in simple terms."],
    sampling_params
)

print("\nModel response:")
print(outputs[0].outputs[0].text)

print("\nGPU memory allocated:",
      round(torch.cuda.memory_allocated()/1024**2, 2), "MiB")


W1116 17:27:09.232000 433 torch/_opaque_base.py:6] torch._opaque_base is deprecated, use torch._custom_class_base instead
W1116 17:27:09.234000 433 torch/_library/opaque_object.py:288] register_opaque_type is deprecated, use register_custom_class instead
W1116 17:27:09.238000 433 torch/_library/opaque_object.py:211] typ='value' is deprecated, use typ='constant' instead


Loading AWQ model on Tesla T4...
(MainProcess pid=433) INFO 11-16 17:27:24 [startup.py:96] non-default args: {'dtype': 'half', 'max_model_len': 512, 'gpu_memory_utilization': 0.75, 'max_num_seqs': 2, 'disable_log_stats': True, 'quantization': 'awq', 'enforce_eager': True, 'model': 'Qwen/Qwen2.5-1.5B-Instruct-AWQ'}


config.json:   0%|          | 0.00/838 [00:00<?, ?B/s]

(MainProcess pid=433) INFO 11-16 17:27:48 [model.py:744] Resolved architecture: Qwen2ForCausalLM
(MainProcess pid=433) INFO 11-16 17:27:48 [model.py:2170] Using max model len 512
(MainProcess pid=433) INFO 11-16 17:27:51 [scheduler.py:310] Chunked prefill is enabled with max_num_batched_tokens=1024.


(MainProcess pid=433) WARNING 11-16 17:27:52 [vllm.py:1778] Enforce eager set, disabling torch.compile, CUDAGraphs, and JIT kernel warmup. This is equivalent to setting -cc.mode=none -cc.cudagraph_mode=none and --kernel_config.enable_jit_warmup=False
(MainProcess pid=433) WARNING 11-16 17:27:52 [vllm.py:1802] Inductor compilation was disabled by user settings, optimizations settings that are only active during inductor compilation will be ignored.
(MainProcess pid=433) INFO 11-16 17:27:52 [kernel.py:480] Final IR op priority after setting platform defaults: IrOpPriorityConfig(rms_norm=['vllm_c', 'native'], fused_add_rms_norm=['vllm_c', 'native'], gelu_and_mul_sparse=['triton', 'native'])
(MainProcess pid=433) INFO 11-16 17:27:52 [vllm.py:2005] Cudagraph is disabled under eager mode
(MainProcess pid=433) INFO 11-16 17:27:52 [compilation.py:317] Enabled custom fusions: norm_quant, act_quant


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 9 files:   0%|          | 0/9 [00:00<?, ?it/s]

(MainProcess pid=433) WARNING 11-16 17:28:00 [system_utils.py:154] We must use the `spawn` multiprocessing start method. Overriding VLLM_WORKER_MULTIPROC_METHOD to 'spawn'. See https://docs.vllm.ai/en/latest/usage/troubleshooting.html#python-multiprocessing for more information. Reasons: CUDA is initialized
(MainProcess pid=433) INFO 11-16 17:29:11 [hf.py:736] Detected the chat template content format to be 'string'. You can set `--chat-template-content-format` to override this.
SUCCESS: AWQ model loaded!


Rendering prompts:   0%|          | 0/1 [00:00<?, ?it/s]

Processed prompts: 100%|██████████| 1/1 [00:06<00:00,  6.06s/it, est. speed input: 1.82 toks/s, output: 10.56 toks/s]


Model response:
 Large language models are like super smart computers that can understand and answer questions in many different languages. Quantization is like making these super smart computers smaller and more compact. It's like shrinking the size of the super smart computer so it can fit in a smaller space, but it still has all the same smart things inside.

GPU memory allocated: 0.0 MiB


In [8]:

import time
import torch
from vllm import LLM, SamplingParams

MODEL = "Qwen/Qwen2.5-1.5B-Instruct-AWQ"

llm = LLM(
    model=MODEL,
    quantization="awq",
    dtype="half",
    max_model_len=512,
    gpu_memory_utilization=0.75,
    max_num_seqs=2,
    enforce_eager=True,
)

prompts = [
    "Explain artificial intelligence in simple terms.",
    "What is machine learning?",
]

params = SamplingParams(
    temperature=0,
    max_tokens=128,
)

torch.cuda.reset_peak_memory_stats()

start = time.perf_counter()
outputs = llm.generate(prompts, params)
elapsed = time.perf_counter() - start

total_tokens = sum(
    len(output.outputs[0].token_ids)
    for output in outputs
)

print("\n========== BENCHMARK ==========")
print(f"Total time: {elapsed:.2f} seconds")
print(f"Generated tokens: {total_tokens}")
print(f"Throughput: {total_tokens / elapsed:.2f} tokens/sec")
print(
    "Peak PyTorch VRAM:",
    round(torch.cuda.max_memory_allocated() / 1024**3, 2),
    "GB"
)

for output in outputs:
    print("\nResponse:", output.outputs[0].text)


(MainProcess pid=433) INFO 11-16 17:32:35 [startup.py:96] non-default args: {'dtype': 'half', 'max_model_len': 512, 'gpu_memory_utilization': 0.75, 'max_num_seqs': 2, 'disable_log_stats': True, 'quantization': 'awq', 'enforce_eager': True, 'model': 'Qwen/Qwen2.5-1.5B-Instruct-AWQ'}
(MainProcess pid=433) INFO 11-16 17:32:35 [model.py:744] Resolved architecture: Qwen2ForCausalLM
(MainProcess pid=433) INFO 11-16 17:32:35 [model.py:2170] Using max model len 512
(MainProcess pid=433) INFO 11-16 17:32:35 [kernel.py:480] Final IR op priority after setting platform defaults: IrOpPriorityConfig(rms_norm=['vllm_c', 'native'], fused_add_rms_norm=['vllm_c', 'native'], gelu_and_mul_sparse=['triton', 'native'])
(MainProcess pid=433) INFO 11-16 17:33:21 [utils.py:628] [shutdown] Process manager: send sigterm to process EngineCore


RuntimeError: Engine core initialization failed. See root cause above. Failed core proc(s): {}

In [9]:
pip install -U vllm pandas

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 24.2 MB/s  0:00:00
  Attempting uninstall: pandas
    Found existing installation: pandas 2.2.3
    Uninstalling pandas-2.2.3:
      Successfully uninstalled pandas-2.2.3
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.3, but you have pandas 3.1.1 which is incompatible.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.
dask-cudf-cu13 26.6.0 requires pandas<2.4.0,>=2.0, but you have pandas 3.1.1 which is incompatible.
cudf-cu13 26.6.0 requires numba<0.65.0,>=0.60.0, but you have numba 0.65.0 which is incompatible.
cudf-cu13 26.6.0 requires pandas<2.4.0,>=2.0, but you have pandas 3.1.1 which is incompatible.
gradio 6.31.0 requires pillow<13.0,>=8.0, but you have pillow 13.0.0 which is incompatible.


In [10]:

import time
import gc
import torch
import pandas as pd

from vllm import LLM, SamplingParams

MODELS = {
    "BEFORE - FP16": "Qwen/Qwen2.5-1.5B-Instruct",
    "AFTER - AWQ": "Qwen/Qwen2.5-1.5B-Instruct-AWQ",
}

PROMPTS = [
    "Explain artificial intelligence in simple terms.",
    "What is machine learning and how does it work?",
    "Explain the difference between AI and deep learning.",
]

PARAMS = SamplingParams(
    temperature=0,
    max_tokens=128,
)

results = []

def benchmark(name, model_name, quantization=None):

    print(f"\n{'='*55}")
    print(f"Testing: {name}")
    print(f"Model: {model_name}")
    print("="*55)

    gc.collect()
    torch.cuda.empty_cache()

    torch.cuda.reset_peak_memory_stats()

    load_start = time.perf_counter()

    llm = LLM(
        model=model_name,
        dtype="half",
        quantization=quantization,
        max_model_len=512,
        gpu_memory_utilization=0.75,
        max_num_seqs=2,
        enforce_eager=True,
    )

    load_time = time.perf_counter() - load_start

    torch.cuda.synchronize()
    torch.cuda.reset_peak_memory_stats()


    llm.generate(
        [PROMPTS[0]],
        SamplingParams(temperature=0, max_tokens=16)
    )

    torch.cuda.synchronize()
    torch.cuda.reset_peak_memory_stats()

    # Benchmark
    start = time.perf_counter()

    outputs = llm.generate(PROMPTS, PARAMS)

    torch.cuda.synchronize()
    inference_time = time.perf_counter() - start

    total_tokens = sum(
        len(output.outputs[0].token_ids)
        for output in outputs
    )

    throughput = total_tokens / inference_time

    peak_memory = (
        torch.cuda.max_memory_allocated() / 1024**3
    )

    result = {
        "Model": name,
        "Load Time (s)": round(load_time, 2),
        "Inference Time (s)": round(inference_time, 2),
        "Tokens Generated": total_tokens,
        "Tokens/sec": round(throughput, 2),
        "Peak PyTorch VRAM (GB)": round(peak_memory, 2),
    }

    results.append(result)

    for i, output in enumerate(outputs):
        print(f"\nPrompt {i+1}: {PROMPTS[i]}")
        print("Response:", output.outputs[0].text)

    print("\nMetrics:", result)

    del llm
    gc.collect()
    torch.cuda.empty_cache()



for name, model in MODELS.items():

    quantization = "awq" if "AWQ" in name else None

    benchmark(
        name,
        model,
        quantization
    )

# Comparison
df = pd.DataFrame(results)

print("\n\n========== BEFORE vs AFTER ==========")
print(df.to_string(index=False))

df.to_csv("fp16_vs_awq_results.csv", index=False)

print("\nSaved: fp16_vs_awq_results.csv")



Testing: BEFORE - FP16
Model: Qwen/Qwen2.5-1.5B-Instruct
(MainProcess pid=433) INFO 11-16 17:34:26 [startup.py:96] non-default args: {'dtype': 'half', 'max_model_len': 512, 'gpu_memory_utilization': 0.75, 'max_num_seqs': 2, 'disable_log_stats': True, 'enforce_eager': True, 'model': 'Qwen/Qwen2.5-1.5B-Instruct'}


config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

(MainProcess pid=433) INFO 11-16 17:34:26 [model.py:744] Resolved architecture: Qwen2ForCausalLM
(MainProcess pid=433) WARNING 11-16 17:34:26 [model.py:2516] Casting torch.bfloat16 to torch.float16.
(MainProcess pid=433) INFO 11-16 17:34:26 [model.py:2170] Using max model len 512
(MainProcess pid=433) INFO 11-16 17:34:27 [kernel.py:480] Final IR op priority after setting platform defaults: IrOpPriorityConfig(rms_norm=['vllm_c', 'native'], fused_add_rms_norm=['vllm_c', 'native'], gelu_and_mul_sparse=['triton', 'native'])


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 9 files:   0%|          | 0/9 [00:00<?, ?it/s]

(MainProcess pid=433) INFO 11-16 17:35:12 [utils.py:628] [shutdown] Process manager: send sigterm to process EngineCore


RuntimeError: Engine core initialization failed. See root cause above. Failed core proc(s): {}

In [11]:

llm = LLM(
    model="Qwen/Qwen2.5-1.5B-Instruct",
    dtype="half",
    max_model_len=512,
    gpu_memory_utilization=0.70,
    max_num_seqs=1,
    enforce_eager=True,
    tensor_parallel_size=1,
)

(MainProcess pid=433) INFO 11-16 17:38:41 [startup.py:96] non-default args: {'dtype': 'half', 'max_model_len': 512, 'gpu_memory_utilization': 0.7, 'max_num_seqs': 1, 'disable_log_stats': True, 'enforce_eager': True, 'model': 'Qwen/Qwen2.5-1.5B-Instruct'}
(MainProcess pid=433) INFO 11-16 17:38:42 [model.py:744] Resolved architecture: Qwen2ForCausalLM
(MainProcess pid=433) WARNING 11-16 17:38:42 [model.py:2516] Casting torch.bfloat16 to torch.float16.
(MainProcess pid=433) INFO 11-16 17:38:42 [model.py:2170] Using max model len 512
(MainProcess pid=433) INFO 11-16 17:38:42 [scheduler.py:310] Chunked prefill is enabled with max_num_batched_tokens=512.
(MainProcess pid=433) INFO 11-16 17:38:42 [kernel.py:480] Final IR op priority after setting platform defaults: IrOpPriorityConfig(rms_norm=['vllm_c', 'native'], fused_add_rms_norm=['vllm_c', 'native'], gelu_and_mul_sparse=['triton', 'native'])
(MainProcess pid=433) INFO 11-16 17:39:08 [utils.py:628] [shutdown] Process manager: send sigterm 

RuntimeError: Engine core initialization failed. See root cause above. Failed core proc(s): {}